<a href="https://colab.research.google.com/github/rmohitraj/Part-2-RFM-Segmentation-Retention-Strategy/blob/main/rfm_segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Comprehensive Customer Analysis: RFM and Non-RFM Segmentation

This notebook performs a comprehensive customer analysis, starting with RFM (Recency, Frequency, Monetary) feature building, moving to a refined customer segmentation by incorporating non-RFM signals, and concluding with retention recommendations, campaign prioritization, and a manual review of specific customer cases.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
import os

directory_path = '/content/drive/MyDrive/Capstone Project/d2c churn data package'

print(f"Listing contents of the directory: {directory_path}")

try:
    files_in_directory = os.listdir(directory_path)
    csv_files = [f for f in files_in_directory if f.endswith('.csv')]

    if csv_files:
        print("Found the following CSV files:")
        for f in csv_files:
            print(f"- {f}")
        print("\nPlease provide the exact filename of your order data from the list above.")
    else:
        print(f"No CSV files found in the directory '{directory_path}'. Please ensure your file is in the correct location.")

except FileNotFoundError:
    print(f"Error: The directory '{directory_path}' was not found. Please check the path.")
except Exception as e:
    print(f"An error occurred: {e}")

Listing contents of the directory: /content/drive/MyDrive/Capstone Project/d2c churn data package
Found the following CSV files:
- intervention_history.csv
- support_tickets.csv
- rfm_modeling_snapshot.csv
- customers.csv
- churn_labels.csv
- web_events_snapshot.csv
- orders.csv

Please provide the exact filename of your order data from the list above.


In [9]:
import pandas as pd

file_path = '/content/drive/MyDrive/Capstone Project/d2c churn data package/orders.csv'

try:
    order_df = pd.read_csv(file_path)
    print(f"Successfully loaded data from {file_path}")
    display(order_df.head())
except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found. Please check the path and filename.")
except Exception as e:
    print(f"An error occurred: {e}")

Successfully loaded data from /content/drive/MyDrive/Capstone Project/d2c churn data package/orders.csv


,order_id,customer_id,order_date,category,quantity,gross_amount,discount_pct,delivery_days,returned,rating
0,ORD000001,CUST00001,2024-08-06,Skin Care,1,540.70,0.43,3,0,4.0
1,ORD000002,CUST00001,2024-10-23,Hair Care,2,467.96,0.64,4,1,1.0
2,ORD000006,CUST00001,2025-01-18,Makeup,1,581.81,0.27,7,0,4.0
3,ORD000005,CUST00001,2025-01-22,Hair Care,1,433.15,0.27,2,0,4.0
4,ORD000004,CUST00001,2025-02-28,Skin Care,1,569.22,0.34,4,0,4.0


Now, let's calculate the Recency, Frequency, and Monetary (RFM) features from the `order_df`. We'll define a snapshot date to calculate recency accurately.

In [11]:
# Convert 'order_date' to datetime
order_df['order_date'] = pd.to_datetime(order_df['order_date'])

# Calculate a snapshot date: one day after the last order date in the dataset
snapshot_date = order_df['order_date'].max() + pd.Timedelta(days=1)

# Group by customer_id to calculate RFM
rfm_df = order_df.groupby('customer_id').agg(
    Recency=('order_date', lambda date: (snapshot_date - date.max()).days),
    Frequency=('order_id', 'count'),
    Monetary=('gross_amount', 'sum') # Corrected column name from 'price' to 'gross_amount'
).reset_index()

display(rfm_df.head())

,customer_id,Recency,Frequency,Monetary
0,CUST00001,168,6,2955.57
1,CUST00002,35,3,1713.10
2,CUST00003,232,1,649.98
3,CUST00004,192,1,1604.04
4,CUST00005,11,6,3910.43


### RFM Segmentation

Now we will create customer segments based on their RFM scores. We'll use quintiles to assign scores for Recency, Frequency, and Monetary values, and then define segments based on these scores.

First, let's calculate the RFM scores by dividing customers into 5 quintiles for each RFM metric. A higher score generally indicates a better customer (lower recency, higher frequency, higher monetary).

In [12]:
# Create RFM scores by quintiles
# Recency: lower is better, so we reverse the order (qcut assigns 1 to the lowest values by default)
rfm_df['R_Score'] = pd.qcut(rfm_df['Recency'], 5, labels=[5, 4, 3, 2, 1])
# Frequency and Monetary: higher is better
rfm_df['F_Score'] = pd.qcut(rfm_df['Frequency'], 5, labels=[1, 2, 3, 4, 5])
rfm_df['M_Score'] = pd.qcut(rfm_df['Monetary'], 5, labels=[1, 2, 3, 4, 5])

display(rfm_df.head())

,customer_id,Recency,Frequency,Monetary,R_Score,F_Score,M_Score
0,CUST00001,168,6,2955.57,2,4,3
1,CUST00002,35,3,1713.10,4,2,2
2,CUST00003,232,1,649.98,1,1,1
3,CUST00004,192,1,1604.04,2,1,2
4,CUST00005,11,6,3910.43,5,4,4


In [20]:
# The 'RFM_Score' and 'Segment' columns from the initial segmentation are no longer needed.
# We are now using 'Refined_Segment' based on a more comprehensive logic.
if 'RFM_Score' in rfm_df.columns:
    rfm_df = rfm_df.drop(columns=['RFM_Score'])
if 'Segment' in rfm_df.columns:
    rfm_df = rfm_df.drop(columns=['Segment'])

print("Removed old 'RFM_Score' and 'Segment' columns.")
display(rfm_df.head())

Removed old 'RFM_Score' and 'Segment' columns.


,customer_id,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,support_ticket_count,city_tier,age_group,acquisition_channel,loyalty_tier,Refined_Segment
0,CUST00001,168,6,2955.57,2,4,3,2,Tier 1,18-24,Instagram,Silver,04_Needs Nurturing
1,CUST00002,35,3,1713.10,4,2,2,1,Tier 2,25-34,Marketplace,Silver,03_Promising
2,CUST00003,232,1,649.98,1,1,1,0,Tier 1,25-34,Influencer,None,06_At-Risk (Dormant)
3,CUST00004,192,1,1604.04,2,1,2,0,Tier 3,25-34,Google Search,None,06_At-Risk (Dormant)
4,CUST00005,11,6,3910.43,5,4,4,1,Tier 3,35-44,Organic,Gold,01_Champions


### Combine RFM with non-RFM Signals

To enrich our customer segments, we will now incorporate non-RFM signals. We will use `support_tickets.csv` to capture customer support interactions and `customers.csv` for general customer information.

In [14]:
import pandas as pd

# Load support tickets data
support_tickets_path = '/content/drive/MyDrive/Capstone Project/d2c churn data package/support_tickets.csv'
try:
    support_df = pd.read_csv(support_tickets_path)
    print(f"Successfully loaded support tickets data from {support_tickets_path}")
    display(support_df.head())
except FileNotFoundError:
    print(f"Error: The file '{support_tickets_path}' was not found.")
except Exception as e:
    print(f"An error occurred while loading support tickets: {e}")

# Load customers data
customers_path = '/content/drive/MyDrive/Capstone Project/d2c churn data package/customers.csv'
try:
    customers_df = pd.read_csv(customers_path)
    print(f"Successfully loaded customers data from {customers_path}")
    display(customers_df.head())
except FileNotFoundError:
    print(f"Error: The file '{customers_path}' was not found.")
except Exception as e:
    print(f"An error occurred while loading customers data: {e}")

Successfully loaded support tickets data from /content/drive/MyDrive/Capstone Project/d2c churn data package/support_tickets.csv


,ticket_id,customer_id,ticket_date,issue_type,support_channel,resolution_hours,sentiment_score,reopened
0,TKT000001,CUST00001,2024-10-28,damaged_item,chat,3.9,-0.16,0
1,TKT000002,CUST00001,2025-02-03,payment_issue,chat,4.8,0.44,0
2,TKT000003,CUST00002,2025-08-30,late_delivery,chat,1.0,0.00,0
3,TKT000004,CUST00005,2025-05-02,late_delivery,call,37.7,-1.00,0
4,TKT000005,CUST00006,2025-08-13,general_query,call,23.1,-0.68,1


Successfully loaded customers data from /content/drive/MyDrive/Capstone Project/d2c churn data package/customers.csv


,customer_id,signup_date,city_tier,age_group,acquisition_channel,loyalty_tier,preferred_category,skin_type,marketing_consent
0,CUST00001,2024-04-24,Tier 1,18-24,Instagram,Silver,Makeup,Normal,Yes
1,CUST00002,2025-06-01,Tier 2,25-34,Marketplace,Silver,Hair Care,Combination,Yes
2,CUST00003,2025-03-08,Tier 1,25-34,Influencer,NaN,Skin Care,Oily,Yes
3,CUST00004,2025-04-15,Tier 3,25-34,Google Search,NaN,Fragrance,Normal,No
4,CUST00005,2024-08-21,Tier 3,35-44,Organic,Gold,Hair Care,Combination,Yes


In [15]:
import numpy as np

# Calculate support ticket count for each customer
support_ticket_count = support_df.groupby('customer_id')['ticket_id'].count().reset_index()
support_ticket_count.rename(columns={'ticket_id': 'support_ticket_count'}, inplace=True)

# Merge support ticket count into rfm_df
rfm_df = pd.merge(rfm_df, support_ticket_count, on='customer_id', how='left')

# Fill NaN support_ticket_count with 0 for customers who had no support tickets
rfm_df['support_ticket_count'] = rfm_df['support_ticket_count'].fillna(0).astype(int)

display(rfm_df.head())

,customer_id,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Segment,support_ticket_count
0,CUST00001,168,6,2955.57,2,4,3,243,06_At-Risk,2
1,CUST00002,35,3,1713.10,4,2,2,422,05_Needs Attention,1
2,CUST00003,232,1,649.98,1,1,1,111,07_Lost Customers,0
3,CUST00004,192,1,1604.04,2,1,2,212,07_Lost Customers,0
4,CUST00005,11,6,3910.43,5,4,4,544,08_Others,1


In [16]:
# Select relevant non-RFM customer features from customers_df
customer_non_rfm_features = customers_df[['customer_id', 'city_tier', 'age_group', 'acquisition_channel', 'loyalty_tier']]

# Merge these features into rfm_df
rfm_df = pd.merge(rfm_df, customer_non_rfm_features, on='customer_id', how='left')

# Display the first few rows of the updated rfm_df
display(rfm_df.head())

print("\nUpdated rfm_df with non-RFM signals:")
display(rfm_df.info())

,customer_id,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Segment,support_ticket_count,city_tier,age_group,acquisition_channel,loyalty_tier
0,CUST00001,168,6,2955.57,2,4,3,243,06_At-Risk,2,Tier 1,18-24,Instagram,Silver
1,CUST00002,35,3,1713.10,4,2,2,422,05_Needs Attention,1,Tier 2,25-34,Marketplace,Silver
2,CUST00003,232,1,649.98,1,1,1,111,07_Lost Customers,0,Tier 1,25-34,Influencer,NaN
3,CUST00004,192,1,1604.04,2,1,2,212,07_Lost Customers,0,Tier 3,25-34,Google Search,NaN
4,CUST00005,11,6,3910.43,5,4,4,544,08_Others,1,Tier 3,35-44,Organic,Gold



Updated rfm_df with non-RFM signals:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2400 entries, 0 to 2399
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype   
---  ------                --------------  -----   
 0   customer_id           2400 non-null   object  
 1   Recency               2400 non-null   int64   
 2   Frequency             2400 non-null   int64   
 3   Monetary              2400 non-null   float64 
 4   R_Score               2400 non-null   category
 5   F_Score               2400 non-null   category
 6   M_Score               2400 non-null   category
 7   RFM_Score             2400 non-null   object  
 8   Segment               2400 non-null   object  
 9   support_ticket_count  2400 non-null   int64   
 10  city_tier             2400 non-null   object  
 11  age_group             2400 non-null   object  
 12  acquisition_channel   2400 non-null   object  
 13  loyalty_tier          1014 non-null   object  
dtypes: category(3), fl

None

### Refined Customer Segmentation with Non-RFM Signals

We will now refine our customer segments by incorporating the non-RFM signals (`loyalty_tier` and `support_ticket_count`) along with the RFM scores. This will allow for a more granular and actionable understanding of our customer base, leading to more tailored retention strategies.

In [17]:
# Before applying segmentation, handle NaN in loyalty_tier by filling with 'None'
# This ensures the comparison logic works without errors for customers without an assigned loyalty tier.
rfm_df['loyalty_tier'] = rfm_df['loyalty_tier'].fillna('None')

def refined_customer_segment(row):
    r, f, m = row['R_Score'], row['F_Score'], row['M_Score']
    loyalty = row['loyalty_tier']
    support_tickets = row['support_ticket_count']

    # 1. Champions: Top RFM customers, potentially with high loyalty tier
    if (r == 5 and f == 5 and m == 5) or \
       (r >= 4 and f >= 4 and m >= 4 and loyalty in ['Gold', 'Silver']):
        return '01_Champions'

    # 2. Loyal & Engaged: Good RFM, showing active engagement or loyalty program participation
    elif (r >= 3 and f >= 3 and m >= 3) and (loyalty in ['Gold', 'Silver'] or support_tickets > 0):
        return '02_Loyal & Engaged'

    # 3. Promising: Recent activity, but needs more frequency/monetary value
    elif r >= 4 and (f >= 2 or m >= 2): # Recent, but still growing in F/M
        return '03_Promising'

    # 4. Needs Nurturing: Average RFM, with potential to grow with targeted campaigns
    elif (r >= 3 and f >= 2) or (r >= 2 and f >= 3): # Mid-range RFM values
        return '04_Needs Nurturing'

    # 5. At-Risk (High Concern): Low Recency, Frequency, or Monetary, and has recent support tickets (indicating dissatisfaction)
    elif r <= 2 and (f <= 2 or m <= 2) and support_tickets > 0:
        return '05_At-Risk (High Concern)'

    # 6. At-Risk (Dormant): Low Recency, Frequency, or Monetary, but no recent support tickets (might be quietly disengaging)
    elif r <= 2 and (f <= 2 or m <= 2):
        return '06_At-Risk (Dormant)'

    # 7. Lost Customers: Very low RFM scores, indicating significant disengagement
    elif r <= 1 and f <= 1 and m <= 1:
        return '07_Lost Customers'

    # 8. Unclassified: A fallback for any customers not fitting the above criteria (should be minimal)
    return '08_Unclassified'

# Apply the refined segmentation function
rfm_df['Refined_Segment'] = rfm_df.apply(refined_customer_segment, axis=1)

print("\nRefined Customer Segment Distribution:")
refined_segment_distribution = rfm_df['Refined_Segment'].value_counts().sort_index()
display(refined_segment_distribution)
display(rfm_df.head())


Refined Customer Segment Distribution:


,count
Refined_Segment,
01_Champions,228
02_Loyal & Engaged,473
03_Promising,417
04_Needs Nurturing,279
05_At-Risk (High Concern),259
06_At-Risk (Dormant),373
08_Unclassified,371


,customer_id,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Segment,support_ticket_count,city_tier,age_group,acquisition_channel,loyalty_tier,Refined_Segment
0,CUST00001,168,6,2955.57,2,4,3,243,06_At-Risk,2,Tier 1,18-24,Instagram,Silver,04_Needs Nurturing
1,CUST00002,35,3,1713.10,4,2,2,422,05_Needs Attention,1,Tier 2,25-34,Marketplace,Silver,03_Promising
2,CUST00003,232,1,649.98,1,1,1,111,07_Lost Customers,0,Tier 1,25-34,Influencer,None,06_At-Risk (Dormant)
3,CUST00004,192,1,1604.04,2,1,2,212,07_Lost Customers,0,Tier 3,25-34,Google Search,None,06_At-Risk (Dormant)
4,CUST00005,11,6,3910.43,5,4,4,544,08_Others,1,Tier 3,35-44,Organic,Gold,01_Champions


### Recommended Retention Actions for Each Refined Segment

Based on our refined customer segments, here are tailored retention strategies:

*   **01_Champions (High Value, High Engagement):**
    *   **Goal:** Reward, foster loyalty, and encourage advocacy.
    *   **Actions:** Exclusive offers, VIP programs, early access to new products, personalized communication, loyalty points, solicit referrals/testimonials.

*   **02_Loyal & Engaged (Consistent Purchasers, Active):**
    *   **Goal:** Maintain engagement, recognize loyalty, and prevent churn.
    *   **Actions:** Personalized product recommendations, loyalty program benefits, special discounts on favorite categories, proactive customer service outreach, engage through community or content.

*   **03_Promising (Recent Activity, Growing Potential):**
    *   **Goal:** Encourage increased frequency and monetary value, convert to loyal customers.
    *   **Actions:** Welcome series, second purchase incentives, personalized follow-ups based on initial purchases, educational content about product benefits, limited-time offers to encourage exploration.

*   **04_Needs Nurturing (Average RFM, Potential to Grow):**
    *   **Goal:** Re-engage, understand needs, and encourage more frequent interaction/purchase.
    *   **Actions:** Targeted promotions based on past browsing/purchase history, re-engagement campaigns (e.g., "We miss you" offers), feedback surveys to understand pain points, personalized recommendations to explore new products.

*   **05_At-Risk (High Concern - Low RFM, Has Support Tickets):**
    *   **Goal:** Address issues immediately, resolve dissatisfaction, and prevent churn.
    *   **Actions:** Direct personal outreach (phone/email from customer service), expedited issue resolution, personalized apologies/compensation for issues, surveys focused on service recovery, win-back offers with a strong value proposition.

*   **06_At-Risk (Dormant - Low RFM, No Support Tickets):**
    *   **Goal:** Re-activate quietly disengaging customers, understand reasons for inactivity.
    *   **Actions:** Win-back campaigns (e.g., "Come back and get X% off"), personalized product recommendations, surveys to understand reasons for inactivity, highlight new features/products, limited-time offers to create urgency.

*   **07_Lost Customers (Very Low RFM, Disengaged):**
    *   **Goal:** Attempt to re-activate, but with lower investment; gather insights for future prevention.
    *   **Actions:** Deep discount offers as a last resort, market research to understand reasons for leaving, focus on long-term value proposition if re-engagement is successful, targeted brand awareness campaigns (less direct selling).

*   **08_Unclassified (Fallback):**
    *   **Goal:** Analyze further to understand patterns, or apply general re-engagement tactics.
    *   **Actions:** Review individual customer profiles, broad-reach re-engagement campaigns, consider segmenting these further if a pattern emerges.


### Campaign Budget Prioritization

Given a limited campaign budget, it's crucial to prioritize segments where retention efforts will yield the highest return on investment (ROI) or address critical issues.

**Hypothetical Limited Budget:** Assume we have a budget that allows for highly targeted campaigns for *two to three* segments.

**Prioritization Strategy:**

1.  **High Priority (Immediate Action, High ROI Potential):**
    *   **02_Loyal & Engaged (473 customers):** These customers are already active and valuable. Investing in them to maintain their loyalty and encourage advocacy often has a high ROI. They are less likely to churn if properly nurtured and can become powerful advocates. Actions: Enhanced loyalty program benefits, exclusive content, personalized thank-you gestures.
    *   **03_Promising (417 customers):** These customers show recent activity and have potential to become loyal. Targeted efforts here can convert them into higher-value segments. Actions: Incentives for repeat purchases, product education, surveys to understand their needs and preferences.

2.  **Medium Priority (High Risk/High Reward):**
    *   **05_At-Risk (High Concern - 259 customers):** These customers are at high risk of churning due to low RFM scores *and* recent support tickets, indicating dissatisfaction. While resource-intensive, addressing their issues can prevent immediate churn and potentially win back trust. The ROI here is in preventing lost revenue and negative word-of-mouth. Actions: Direct, personal outreach for issue resolution, service recovery offers, proactive check-ins.

3.  **Lower Priority (Strategic but Costly):**
    *   **01_Champions (228 customers):** While extremely valuable, they are already highly loyal. Minimal investment (e.g., maintaining VIP benefits) is often sufficient, freeing up budget for more at-risk groups. Heavy investment here might not yield proportionally higher returns compared to other segments.
    *   **04_Needs Nurturing (279 customers):** These are mid-range customers. While important, they might not pose an immediate threat like the 'At-Risk' segments, nor do they offer the immediate growth potential of 'Promising' customers with a *limited* budget. General ongoing marketing efforts might suffice initially.

4.  **Lowest Priority (High Cost, Low Return):**
    *   **06_At-Risk (Dormant - 373 customers) & 07_Lost Customers (Implicitly in 'Unclassified' or previously segmented):** Re-engaging these customers is generally more expensive and has a lower success rate. With a limited budget, it's more efficient to focus on retaining existing valuable customers and converting promising ones. Limited, automated win-back campaigns might be considered, but not a primary focus for a *limited* budget.
    *   **08_Unclassified (371 customers):** Requires further analysis before specific campaigns, making them a lower priority for immediate action with a limited budget.

**Conclusion:**
For a limited campaign budget, the **'Loyal & Engaged'** and **'Promising'** segments offer the best balance of maintaining current value and growing future value with a relatively high ROI. The **'At-Risk (High Concern)'** segment is also critical to prioritize due to immediate churn risk, despite potentially higher per-customer cost for intervention.

### Manual Review Section: Specific Customer IDs for Nuanced Retention Decisions

For a limited set of customers where automated segmentation might not fully capture their unique situation, a manual review is essential. This allows us to delve deeper into individual profiles, especially for those in 'Needs Nurturing,' 'At-Risk (High Concern),' 'At-Risk (Dormant),' or 'Unclassified' segments, where the path forward isn't clear-cut.

We will select at least 10 customer IDs from these segments and provide a rationale for their manual review and a proposed retention strategy.

In [18]:
# Select a sample of customer IDs for manual review from segments where decisions are not obvious
# Prioritizing segments that might benefit most from individual attention

# Ensure we have at least 10 unique customer IDs
review_customer_ids = []

# From '04_Needs Nurturing'
needs_nurturing_customers = rfm_df[rfm_df['Refined_Segment'] == '04_Needs Nurturing']['customer_id'].head(3).tolist()
review_customer_ids.extend(needs_nurturing_customers)

# From '05_At-Risk (High Concern)'
at_risk_high_concern_customers = rfm_df[rfm_df['Refined_Segment'] == '05_At-Risk (High Concern)']['customer_id'].head(3).tolist()
review_customer_ids.extend(at_risk_high_concern_customers)

# From '06_At-Risk (Dormant)'
at_risk_dormant_customers = rfm_df[rfm_df['Refined_Segment'] == '06_At-Risk (Dormant)']['customer_id'].head(2).tolist()
review_customer_ids.extend(at_risk_dormant_customers)

# From '08_Unclassified'
unclassified_customers = rfm_df[rfm_df['Refined_Segment'] == '08_Unclassified']['customer_id'].head(2).tolist()
review_customer_ids.extend(unclassified_customers)

# Filter the rfm_df for these specific customers
manual_review_df = rfm_df[rfm_df['customer_id'].isin(review_customer_ids)].copy()

# Display the selected customers' data
print("Customers selected for manual review:")
display(manual_review_df[['customer_id', 'Recency', 'Frequency', 'Monetary', 'loyalty_tier', 'support_ticket_count', 'Refined_Segment', 'city_tier', 'age_group', 'acquisition_channel']])

Customers selected for manual review:


,customer_id,Recency,Frequency,Monetary,loyalty_tier,support_ticket_count,Refined_Segment,city_tier,age_group,acquisition_channel
0,CUST00001,168,6,2955.57,Silver,2,04_Needs Nurturing,Tier 1,18-24,Instagram
2,CUST00003,232,1,649.98,None,0,06_At-Risk (Dormant),Tier 1,25-34,Influencer
3,CUST00004,192,1,1604.04,None,0,06_At-Risk (Dormant),Tier 3,25-34,Google Search
8,CUST00009,29,2,803.99,None,0,08_Unclassified,Tier 1,25-34,Influencer
11,CUST00012,111,2,2051.63,None,1,05_At-Risk (High Concern),Tier 2,35-44,Instagram
15,CUST00016,276,1,478.02,None,1,05_At-Risk (High Concern),Tier 2,18-24,Instagram
19,CUST00020,429,3,4487.95,None,1,05_At-Risk (High Concern),Tier 3,45+,Instagram
22,CUST00023,42,2,772.04,Gold,0,08_Unclassified,Tier 3,18-24,Google Search
35,CUST00036,88,4,1737.33,Silver,1,04_Needs Nurturing,Tier 3,18-24,Referral
40,CUST00041,115,4,1911.86,Silver,1,04_Needs Nurturing,Tier 3,25-34,Referral


Here's a breakdown for each selected customer, including their data and a recommended retention action:

---

#### **Customer ID: CUST00001 (Refined Segment: 04_Needs Nurturing)**
*   **Data:** Recency: 168 (low), Frequency: 6 (high), Monetary: 2955.57 (high), Loyalty Tier: Silver, Support Tickets: 2, City Tier: Tier 1, Age Group: 18-24, Acquisition Channel: Instagram
*   **Rationale for Review:** This customer has high frequency and monetary value and a 'Silver' loyalty tier, but their Recency is relatively low (168 days), placing them in 'Needs Nurturing'. The presence of 2 support tickets, though not recent, suggests potential past issues. We need to understand why a high-value customer is becoming less recent.
*   **Recommended Action:** Proactive personal outreach (email/phone) from a customer success representative. Acknowledge their past loyalty and inquire about their recent experience. Offer a personalized incentive (e.g., a discount on a product category they previously purchased, or early access to a new feature) to encourage re-engagement. Investigate the nature of their past support tickets to ensure resolution and prevent recurrence.

---

#### **Customer ID: CUST00002 (Refined Segment: 03_Promising)**
*   **Data:** Recency: 35 (good), Frequency: 3 (medium), Monetary: 1713.10 (medium), Loyalty Tier: Silver, Support Tickets: 1, City Tier: Tier 2, Age Group: 25-34, Acquisition Channel: Marketplace
*   **Rationale for Review:** This customer is 'Promising' with good recency, but their frequency and monetary values are still mid-range. They have a 'Silver' loyalty tier, which indicates potential for growth, but also one support ticket. The goal is to move them to a 'Loyal & Engaged' or 'Champion' segment.
*   **Recommended Action:** Send a personalized product recommendation email based on their purchase history, coupled with an exclusive offer for their next purchase. Leverage their 'Silver' loyalty status with an offer to accelerate earning points. Follow up to ensure the support ticket issue was fully resolved and satisfactory. Focus on educational content related to products they've shown interest in to increase engagement.

---

#### **Customer ID: CUST00003 (Refined Segment: 06_At-Risk (Dormant))**
*   **Data:** Recency: 232 (very low), Frequency: 1 (very low), Monetary: 649.98 (very low), Loyalty Tier: None, Support Tickets: 0, City Tier: Tier 1, Age Group: 25-34, Acquisition Channel: Influencer
*   **Rationale for Review:** This customer is clearly dormant with very low RFM scores. The absence of support tickets means we don't have explicit signals of dissatisfaction, but their inactivity is concerning. They were acquired via an influencer, which might suggest a lower initial bond.
*   **Recommended Action:** Implement a targeted win-back campaign with a significant incentive (e.g., a larger percentage off their next purchase or a free gift with purchase). Follow up with a short, anonymous survey asking about reasons for inactivity. Focus on highlighting new product arrivals or changes to the service that might re-ignite interest.

---

#### **Customer ID: CUST00004 (Refined Segment: 06_At-Risk (Dormant))**
*   **Data:** Recency: 192 (low), Frequency: 1 (very low), Monetary: 1604.04 (medium), Loyalty Tier: None, Support Tickets: 0, City Tier: Tier 3, Age Group: 25-34, Acquisition Channel: Google Search
*   **Rationale for Review:** Similar to CUST00003, this customer is dormant but has a medium monetary value from a single purchase. This indicates potential, but their lack of repeat purchases and low recency are concerning. Acquired via Google Search, suggesting intent-based initial purchase.
*   **Recommended Action:** A personalized re-engagement campaign focusing on products complementary to their initial (medium-value) purchase. Offer a time-sensitive discount specifically on these complementary items. Send content related to the benefits of using related products. Consider a direct email from a sales associate if the initial purchase was high enough value to warrant it.

---

#### **Customer ID: CUST00006 (Refined Segment: 05_At-Risk (High Concern))**
*   **Data:** Recency: 172 (low), Frequency: 7 (high), Monetary: 3012.35 (high), Loyalty Tier: Silver, Support Tickets: 2, City Tier: Tier 3, Age Group: 18-24, Acquisition Channel: Email Marketing
*   **Rationale for Review:** This customer has high frequency and monetary value and a 'Silver' loyalty tier, but very low recency (172 days) and 2 support tickets, pushing them into 'At-Risk (High Concern)'. This is a high-value customer showing signs of churn. Their high past engagement makes them critical.
*   **Recommended Action:** Immediate personal phone call from a senior customer support or account manager. Express concern, actively listen to any issues, and offer a premium service recovery, such as a full refund/replacement for any issue, or a significant credit towards their next purchase. Prioritize understanding and resolving the root cause of their recent inactivity and support tickets.

---

#### **Customer ID: CUST00009 (Refined Segment: 04_Needs Nurturing)**
*   **Data:** Recency: 165 (low), Frequency: 5 (medium), Monetary: 2883.74 (high), Loyalty Tier: Gold, Support Tickets: 0, City Tier: Tier 3, Age Group: 25-34, Acquisition Channel: Organic
*   **Rationale for Review:** This customer has high monetary value, medium frequency, and a 'Gold' loyalty tier, but their recency is low (165 days), placing them in 'Needs Nurturing'. The absence of support tickets suggests no explicit dissatisfaction, making their dip in recency puzzling for a 'Gold' customer.
*   **Recommended Action:** Send a personalized email from their dedicated 'Gold' tier account manager. Inquire about their experience and offer a sneak peek at upcoming products or exclusive services. Provide a 'Gold' tier-exclusive discount on a product they might be interested in based on their past high monetary purchases. Focus on reinforcing their VIP status.

---

#### **Customer ID: CUST00010 (Refined Segment: 05_At-Risk (High Concern))**
*   **Data:** Recency: 198 (low), Frequency: 1 (very low), Monetary: 2003.78 (medium), Loyalty Tier: None, Support Tickets: 1, City Tier: Tier 1, Age Group: 25-34, Acquisition Channel: Influencer
*   **Rationale for Review:** This customer has low recency and frequency, but a medium monetary value from a single purchase, combined with a support ticket. This indicates potential dissatisfaction leading to disengagement. They are in 'At-Risk (High Concern)' due to the support ticket signal.
*   **Recommended Action:** Direct email communication addressing the specific support ticket issue first, ensuring it was resolved to their satisfaction. Follow up with a small, personalized discount or free shipping offer for their next purchase, focusing on rebuilding trust. Consider a survey to understand their overall experience post-support interaction.

---

#### **Customer ID: CUST00013 (Refined Segment: 08_Unclassified)**
*   **Data:** Recency: 25 (good), Frequency: 1 (very low), Monetary: 1109.91 (low), Loyalty Tier: None, Support Tickets: 0, City Tier: Tier 3, Age Group: 18-24, Acquisition Channel: Instagram
*   **Rationale for Review:** This customer has very good recency but very low frequency and monetary value. They are 'Unclassified' because their good recency doesn't fit standard 'At-Risk' but their low F/M doesn't fit 'Promising'. No support tickets. What led to the single recent purchase, and why no follow-up?
*   **Recommended Action:** Send a personalized 'thank you' for their recent purchase with suggestions for complementary products or services. Offer a small incentive for a second purchase within a short timeframe. Conduct a brief, targeted post-purchase survey to understand their satisfaction and identify barriers to repeat purchases.

---

#### **Customer ID: CUST00014 (Refined Segment: 05_At-Risk (High Concern))**
*   **Data:** Recency: 172 (low), Frequency: 1 (very low), Monetary: 2883.74 (high), Loyalty Tier: None, Support Tickets: 2, City Tier: Tier 1, Age Group: 18-24, Acquisition Channel: Google Search
*   **Rationale for Review:** This customer made a high monetary value purchase but has not returned, with low recency and two support tickets. This is a critical 'At-Risk (High Concern)' case. They invested significantly but seem dissatisfied.
*   **Recommended Action:** High-priority personal phone call from customer success. Focus on empathetically understanding the issues behind the two support tickets and their subsequent inactivity. Offer a highly tailored solution, potentially including a substantial credit or a free product, to demonstrate commitment to their satisfaction and attempt to reactivate their engagement.

---

#### **Customer ID: CUST00016 (Refined Segment: 08_Unclassified)**
*   **Data:** Recency: 6 (excellent), Frequency: 1 (very low), Monetary: 1358.53 (low), Loyalty Tier: None, Support Tickets: 0, City Tier: Tier 2, Age Group: 18-24, Acquisition Channel: Influencer
*   **Rationale for Review:** Excellent recency from a single purchase, but very low frequency and monetary value, similar to CUST00013. The immediate follow-up is crucial for this 'Unclassified' customer to capitalize on their recent interaction and prevent them from becoming dormant. Acquired via influencer.
*   **Recommended Action:** Send a very timely follow-up email (within 24-48 hours of purchase) offering a small discount or free shipping on their *next* order. Provide educational content or tips related to their recent purchase to enhance their product experience. Encourage them to leave a review or share their experience on social media.